# 05. Lyrics Preprocessing

1. Data Loading
2. Non-Lyric Marker Inspection
3. Text Cleaning
   - 3.1 Cleaning Rules
   - 3.2 Before/After Samples
   - 3.3 Remaining Marker Check
4. Non-Korean Version Filter
5. Tokenization and Language Tagging
   - 5.1 Tokenization
   - 5.2 Filler Flag
   - 5.3 Token Summary
6. Export

## 1. Data Loading

### Objective
Load the labeled song-level dataset from `04_generation_labeling.ipynb`.

In [1]:
from pathlib import Path
import re
import unicodedata

import pandas as pd

INTERIM_DIR = Path("../data/interim")

songs = pd.read_pickle(INTERIM_DIR / "songs_labeled.pkl")

print(f"Songs: {len(songs):,}")
print(f"Lyric lines: {songs['lyrics'].apply(len).sum():,}")

Songs: 7,988
Lyric lines: 443,320


## 2. Non-Lyric Marker Inspection

### Objective
Measure how often non-lyric markers appear in the raw lyrics before cleaning.

### Procedure
Count lines containing each marker type:
- **Square brackets:** `[Verse 1]`, `[RM]`
- **Section keywords:** verse, chorus, hook, bridge, intro, outro, rap, etc.
- **Member labels:** `RM)`, `지민:` at the start of a line
- **Repeat markers:** `*`, `#`, `※`, `반복`, `repeat`, `x2`

In [2]:
SECTION_WORDS = (
    r"verse|chorus|pre-?chorus|post-?chorus|semi-?hook|hook|bridge|intro|outro|interlude|refrain|"
    r"rap|narration|naration|music|all|repeat|"
    r"후렴|반복|랩|나레이션|간주|전주|다같이|모두|합창|코러스|브릿지"
)
REPEAT_TOKEN = r"(?<!\w)(?:[x×]\s?\d+|\d+\s?[x×])(?!\w)|[*#※]+"

# Section keyword alone in parentheses, optionally with a number, repeat count, or short name: (Chorus), (Rap: RM), (반복 x2)
SECTION_PAREN = rf"[(]\s*(?:{SECTION_WORDS})\b[\s\d:.\-x×*]*(?:[:\-]\s*[^)]{{0,12}})?[)]"
# Line made only of section keywords, numbers, and symbols: Chorus, Verse 2:, *후렴 반복
SECTION_LINE = rf"^(?:[\W\d_]*(?:{SECTION_WORDS}|x\d+|\d+x)\b)+[\W\d_]*$"

MARKER_PATTERNS = {
    "square_brackets": r"\[[^\]]*\]",
    "section_keyword": rf"(?i){SECTION_PAREN}|{SECTION_LINE}",
    "member_label": r"^\s*[^\s()\[\]:]{1,12}\s*[):]\s+\S",
    "repeat_marker": rf"(?i){REPEAT_TOKEN}",
}

lines = songs[["song_key", "lyrics"]].explode("lyrics").dropna()
lines["lyrics"] = lines["lyrics"].astype(str)

marker_counts = pd.DataFrame({
    name: {
        "lines": lines["lyrics"].str.contains(p, regex=True).sum(),
        "songs": lines.loc[lines["lyrics"].str.contains(p, regex=True), "song_key"].nunique(),
    }
    for name, p in MARKER_PATTERNS.items()
}).T
marker_counts["songs (%)"] = (marker_counts["songs"] / len(songs) * 100).round(2)
display(marker_counts)

for name, p in MARKER_PATTERNS.items():
    sample = lines.loc[lines["lyrics"].str.contains(p, regex=True), "lyrics"].head(5).tolist()
    print(f"\n[{name}]")
    for s in sample:
        print("  ", s)

,lines,songs,songs (%)
square_brackets,69,7,0.09
section_keyword,79,38,0.48
member_label,37,7,0.09
repeat_marker,136,50,0.63



[square_brackets]
   이밤에 취해 [술에 취해] 흔들리고 있네요.
   이밤에 취해 [술에 취해] 떠나고만 싶네요.
   이밤에 취해 [술에 취해] 흔들리고 있네요
   딱 한 번만이라도 [가지마라 가지마라]
   날 위해 웃어준다면 [나를두고 떠나가지마라]

[section_keyword]
   모두 모두 모두 모두
   (rap) 이제 멈춰 제발 멈춰 너의 쓸데 없는 무관심 차가움 이젠 제
   *반복
   (Music) 제대로 (Rap) 제대로
   chorus

[member_label]
   *반복) 어쩌면 너는 모두 잊었을지도 몰라
   가버려) 눈물 씻고 화장하고 다른 남잘 찾겠어
   RYAN) 천사 같았어 날개만 없었지
   ACE) 나를 보는 너의 눈 난 그대로 얼어서
   P.O,NEO) 너무 많은 나의 바램이 네 귀를 막아

[repeat_marker]
   5:469]#@6준비 uh 시작
   5:469]#@6아침에 일어나서
   *내게 전해줘 그대 사랑을
   이제 더이상 난 기다릴 수 없어요.*
   *반복


## 3. Text Cleaning

### 3.1 Cleaning Rules

#### Procedure
Applied to each line in this order:
1. **Unicode normalization:** NFKC (full-width characters to half-width), unify quotes (`’ ‘ ´ ` → `'`), collapse whitespace. Remove encoding residue (`5:469]#@6`) and censored words (`bulls**t`, `f***ing`).
2. **Square brackets with labels:** remove when they hold a section keyword or a single short word such as a member name (`[Verse 1]`, `[형준]`, `[All]`). Other square brackets hold backing vocals (`[가지마라 가지마라]`); only the brackets are removed.
3. **Section keywords in parentheses:** remove entirely when the parentheses hold only the label (`(Chorus)`, `(Rap: RM)`, `(반복 x2)`). Ad-libs that start with the same word, such as `(All I want)` or `(모두 다 사랑해)`, are kept.
4. **Section-only lines:** drop lines that contain only a section keyword (`Chorus`, `Verse 2:`, `후렴`).
5. **Line-initial labels:** remove a section keyword at the start of a lyric line (`verse 1 가사`, `hook 가사`) and a short member label followed by `)` or `:` (`RM) 가사`, `지민: 가사`).
6. **Repeat markers:** remove `*`, `#`, `※`, `x2`, `2x`. The words `반복` and `repeat` are removed only as labels (rules 3–4), so lyrics such as `반복되는 하루` are kept.
7. **Other parentheses and brackets (ad-libs, backing vocals):** keep the content and remove only the brackets (`(Oh yeah)` → `Oh yeah`).
8. **Lowercase** Latin characters; drop empty lines.

Repeated sections are kept, since they reflect the actual sung lyrics.

In [3]:
QUOTE_MAP = str.maketrans({"’": "'", "‘": "'", "´": "'", "`": "'", "“": '"', "”": '"'})

# Square brackets: remove when they hold a label (section keyword or a single short word such as a member name)
RE_SQUARE_LABEL = re.compile(rf"(?i)\[\s*(?:(?:{SECTION_WORDS})\b[^\]]{{0,15}}|[^\s\[\]]{{1,10}})\s*\]")
# Section label at the start of a lyric line: "verse 1 가사", "hook: 가사"
RE_SECTION_PREFIX = re.compile(
    r"(?i)^\W*(?:verse|chorus|pre-?chorus|post-?chorus|semi-?hook|hook|bridge|intro|outro|interlude|refrain|narr?ation)"
    r"(?:\s*\d+\s*[:.\-)]?\s*|\s*[:.\-)]\s*|\s+(?=[\uac00-\ud7a3]))"
)  # removed only when followed by a number, a colon, or Korean text, so "bridge over ..." is kept
RE_CENSORED = re.compile(r"(?i)[a-z]+\*+[a-z]*")          # bulls**t, f***ing
RE_ARTIFACT = re.compile(r"\d+:\d+\][#@]*\d*")          # encoding residue such as 5:469]#@6
RE_SECTION_PAREN = re.compile(rf"(?i){SECTION_PAREN}")
RE_SECTION_LINE = re.compile(rf"(?i){SECTION_LINE}")
RE_MEMBER = re.compile(r"^\s*[^\s()\[\]:]{1,12}\s*[):]\s+(?=\S)")
RE_REPEAT = re.compile(rf"(?i){REPEAT_TOKEN}")
RE_PAREN = re.compile(r"[()\[\]]")
RE_SPACE = re.compile(r"\s+")

def clean_line(line):
    s = unicodedata.normalize("NFKC", str(line)).translate(QUOTE_MAP)
    s = RE_ARTIFACT.sub(" ", s)
    s = RE_CENSORED.sub(" ", s)
    s = RE_SQUARE_LABEL.sub(" ", s)
    s = RE_SECTION_PAREN.sub(" ", s)
    if RE_SECTION_LINE.match(s):
        return ""
    s = RE_SECTION_PREFIX.sub("", s)
    s = RE_MEMBER.sub("", s)
    s = RE_REPEAT.sub(" ", s)
    s = RE_PAREN.sub(" ", s)
    s = RE_SPACE.sub(" ", s).strip().lower()
    return s

def clean_lyrics(lines):
    out = [clean_line(l) for l in lines]
    return [l for l in out if re.search(r"\w", l)]

songs["lyrics_clean"] = songs["lyrics"].apply(clean_lyrics)
songs["n_lines_raw"] = songs["lyrics"].apply(len)
songs["n_lines_clean"] = songs["lyrics_clean"].apply(len)

print(f"Lines before: {songs['n_lines_raw'].sum():,}")
print(f"Lines after:  {songs['n_lines_clean'].sum():,} "
      f"({1 - songs['n_lines_clean'].sum() / songs['n_lines_raw'].sum():.2%} removed)")
print(f"Songs with no lines left: {(songs['n_lines_clean'] == 0).sum()}")

Lines before: 443,320
Lines after:  419,986 (5.26% removed)
Songs with no lines left: 0


### 3.2 Before/After Samples

#### Objective
Manually check the cleaning result on lines that contained markers.

#### Procedure
- Show 40 random raw lines that matched any marker pattern, with their cleaned version.
- Re-run with a different `random_state` to see other samples.

In [4]:
any_marker = "(?i)" + "|".join(f"(?:{p.replace('(?i)', '')})" for p in MARKER_PATTERNS.values())
marked = lines[lines["lyrics"].str.contains(any_marker, regex=True)]

sample = marked.sample(min(40, len(marked)), random_state=0).copy()
sample["cleaned"] = sample["lyrics"].apply(clean_line)

pd.set_option("display.max_colwidth", 80)
display(sample[["lyrics", "cleaned"]].rename(columns={"lyrics": "raw"}))

,raw,cleaned
5633,개똥 같은 소리하네 bulls**t,개똥 같은 소리하네
6681,And her *** is soft,and her is soft
7786,[Post-Chorus],
524,이제 더이상 난 기다릴 수 없어요.*,이제 더이상 난 기다릴 수 없어요.
1904,[All] We’ll unlock the door forever,we'll unlock the door forever
7645,그럼 걸리는 거야 *밥병,그럼 걸리는 거야 밥병
7870,I’m still f***ing christian,i'm still christian
3276,택연) 넌 나만의 Star 아무도 아무것도 몰라 너는 걱정마,넌 나만의 star 아무도 아무것도 몰라 너는 걱정마
3769,X2,
1985,[규종] 시간이 다가왔어 나 지금이야,시간이 다가왔어 나 지금이야


### 3.3 Remaining Marker Check

#### Objective
Find markers that the rules missed.

#### Procedure
- Count cleaned lines that still look like labels: section keywords, a colon, or a repeat count.
- Generic words that also appear in real lyrics (`all`, `rap`, `모두`, `반복`) are not checked here.
- If a pattern appears often, add it to the rules in 3.1 and re-run.

In [5]:
clean_lines = songs[["song_key", "lyrics_clean"]].explode("lyrics_clean").dropna()

REMAINING = {
    "section keyword": r"(?i)\b(?:verse|chorus|hook|bridge|intro|outro|interlude|refrain|narration)\b|후렴|나레이션|간주|전주|브릿지",
    "colon": r":",
    "repeat-like": r"(?i)\b\d\s?x\b|\bx\s?\d\b",
}
for name, p in REMAINING.items():
    hits = clean_lines[clean_lines["lyrics_clean"].str.contains(p, regex=True)]
    print(f"[{name}] {len(hits):,} lines")
    for s in hits["lyrics_clean"].head(8):
        print("  ", s)

[section keyword] 34 lines
   with me intro
   we can hook up at the store
   take it back to the interlude
   leave your fate to the u-n-i verse
   take em' to the chorus
   take em' to the bridge
   take em' to the chorus
   take em' to the chorus
[colon] 18 lines
   8:45 그대는 하늘나라로
   평온한 세계로 8:45
   8:45 그대는 하늘나라로
   평온한 세계로 8:45
   8:45 그대는
   8:45 그대는 하늘나라로
   8:45 그대는 하늘나라로
   8:45 그대는 하늘나라로
[repeat-like] 0 lines


## 4. Non-Korean Version Filter

### Objective
Exclude songs whose lyrics are mostly Japanese or Chinese (e.g. Japanese versions of K-pop songs).

### Procedure
- Count characters by script in the cleaned lyrics: Hangul, Latin, Japanese kana, Chinese characters (Han).
- `kana_ratio` = kana / all letters; `han_ratio` = Han / all letters.
- Exclude a song if `kana_ratio` ≥ 0.5 (mostly Japanese) or `han_ratio` ≥ 0.5 (mostly Chinese).
- Songs with only a few Japanese or Chinese lines are kept.

In [6]:
SCRIPTS = {
    "hangul": r"[\uac00-\ud7a3\u3131-\u318e]",
    "latin": r"[a-z]",
    "kana": r"[\u3040-\u30ff]",
    "han": r"[\u4e00-\u9fff]",
}

text = songs["lyrics_clean"].str.join(" ")
for name, p in SCRIPTS.items():
    songs[f"n_{name}"] = text.str.count(p)

n_letters = songs[[f"n_{k}" for k in SCRIPTS]].sum(axis=1).replace(0, pd.NA)
songs["kana_ratio"] = songs["n_kana"] / n_letters
songs["han_ratio"] = songs["n_han"] / n_letters

songs["flag_japanese"] = songs["kana_ratio"].fillna(0) >= 0.5
songs["flag_chinese"] = songs["han_ratio"].fillna(0) >= 0.5
songs["flag_no_lyrics"] = songs["n_lines_clean"] == 0
songs["exclude_lyrics"] = songs["flag_japanese"] | songs["flag_chinese"] | songs["flag_no_lyrics"]

display(songs[["flag_japanese", "flag_chinese", "flag_no_lyrics", "exclude_lyrics"]].sum().to_frame("songs"))

print("Songs with some Japanese (kana_ratio > 0):", (songs["kana_ratio"].fillna(0) > 0).sum())
display(
    songs[songs["exclude_lyrics"]]
    [["song_name", "artist", "release_year", "kana_ratio", "han_ratio"]]
    .round(2).head(20)
)

,songs
flag_japanese,4
flag_chinese,0
flag_no_lyrics,0
exclude_lyrics,4


Songs with some Japanese (kana_ratio > 0): 11


,song_name,artist,release_year,kana_ratio,han_ratio
999,Yuki No Hana,Nakashima Mika,2004,0.76,0.24
1049,Dive To Blue,L'Arc-en-Ciel,2001,0.70,0.30
7919,愛を伝えたいだとか / Ai Wo Tsutaetaidatoka (사랑을 전하고 싶다든가),aimyon,2017,0.77,0.23
7933,NIGHT DANCER (BIG Naughty Remix),imase,2023,0.56,0.14


## 5. Tokenization and Language Tagging

### 5.1 Tokenization

#### Procedure
- Split each line on whitespace, then on script boundaries, so that mixed words such as `love해` become `love` + `해`.
- Token types and language tags:
  - `ko`: Hangul run
  - `en`: Latin run, including apostrophes inside words (`don't`, `i'm`)
  - `ja`: kana run; `zh`: Han run; `num`: digits
- Punctuation is dropped. Hyphenated words are split (`la-la` → `la`, `la`).
- Language is tagged by script, which is more reliable than a language-ID model for single tokens. py3langid is used later at the song level as a supplementary check.

In [7]:
RE_TOKEN = re.compile(
    r"(?P<ko>[\uac00-\ud7a3\u3131-\u318e]+)"
    r"|(?P<en>[a-z]+(?:'[a-z]+)*)"
    r"|(?P<ja>[\u3040-\u30ff]+)"
    r"|(?P<zh>[\u4e00-\u9fff]+)"
    r"|(?P<num>[0-9]+)"
)

def tokenize_song(song_key, lines):
    rows = []
    for line_no, line in enumerate(lines):
        for token_no, m in enumerate(RE_TOKEN.finditer(line)):
            rows.append((song_key, line_no, token_no, m.group(), m.lastgroup))
    return rows

kept = songs[~songs["exclude_lyrics"]]
tokens = pd.DataFrame(
    [r for k, l in zip(kept["song_key"], kept["lyrics_clean"]) for r in tokenize_song(k, l)],
    columns=["song_key", "line_no", "token_no", "token", "lang"],
)

print(f"Tokens: {len(tokens):,} from {tokens['song_key'].nunique():,} songs")
display(tokens["lang"].value_counts().to_frame("tokens"))
display(tokens.head(20))

Tokens: 1,671,530 from 7,984 songs


,tokens
lang,
ko,1273499
en,393494
num,3569
ja,506
zh,462


,song_key,line_no,token_no,token,lang
0,38541,0,0,다가오지마,ko
1,38541,0,1,더,ko
2,38541,0,2,이상,ko
3,38541,1,0,위로될,ko
4,38541,1,1,수,ko
5,38541,1,2,없다는,ko
6,38541,1,3,것을,ko
7,38541,2,0,모르는,ko
8,38541,2,1,것처럼,ko
9,38541,3,0,걱정하지마,ko


### 5.2 Filler Flag

#### Objective
Mark English vocables and fillers so that the linguistic features can be computed with and without them.

#### Procedure
- Fillers: `oh`, `ooh`, `ah`, `uh`, `yeah`, `ya`, `hey`, `ho`, `woo`, `wow`, `whoa`, `la`, `na`, `da`, `ha`, `mm`, `hmm`, `eh`, `ay`, and their stretched or repeated forms (`ohhh`, `yeahhh`, `lalala`, `hahaha`).
- Stretched letters are collapsed before matching (`ohhhh` → `ohh`).
- `baby` is treated as a lexical word, not a filler.

In [8]:
FILLER_PATTERN = re.compile(
    r"^(?:"
    r"o+h*|o+o+h*|a+h+|u+h+|uh-?huh|m+|h?m+|e+h+|a+y+|"
    r"y+e+a*h*|y+a+h*|y+e+|h+e+y+|h+o+|w+o+o*|w+o+w+|w+h*o+a+|"
    r"(?:la)+|(?:na)+|(?:da)+|(?:du)+|(?:ba)+|(?:doo)+|(?:ooh)+|(?:ha)+|huh"
    r")$"
)

def is_filler(token):
    t = re.sub(r"(.)\1{2,}", r"\1\1", token)
    return bool(FILLER_PATTERN.match(t)) or bool(FILLER_PATTERN.match(token))

en_mask = tokens["lang"] == "en"
tokens["is_filler"] = False
tokens.loc[en_mask, "is_filler"] = tokens.loc[en_mask, "token"].map(is_filler)

print(f"English tokens: {en_mask.sum():,}")
print(f"Filler tokens: {tokens['is_filler'].sum():,} ({tokens['is_filler'].sum() / en_mask.sum():.2%} of English)")

print("\nMost frequent fillers:")
display(tokens[tokens["is_filler"]]["token"].value_counts().head(20).to_frame("count"))

print("\nMost frequent English non-filler tokens (check for missed fillers):")
display(tokens[en_mask & ~tokens["is_filler"]]["token"].value_counts().head(40).to_frame("count"))

English tokens: 393,494
Filler tokens: 29,592 (7.52% of English)

Most frequent fillers:


,count
token,
oh,9213
yeah,3858
la,1964
hey,1769
uh,1724
ah,1418
na,1010
ooh,952
woo,919



Most frequent English non-filler tokens (check for missed fillers):


,count
token,
you,18288
i,15404
me,9547
my,7846
love,7821
the,6596
it,6229
baby,5898
to,4894


### 5.3 Token Summary

#### Objective
Summarize token counts per song as a sanity check before feature extraction.

In [9]:
per_song = tokens.groupby("song_key").agg(
    n_tokens=("token", "size"),
    n_ko=("lang", lambda s: (s == "ko").sum()),
    n_en=("lang", lambda s: (s == "en").sum()),
    n_en_filler=("is_filler", "sum"),
)
per_song["en_share"] = per_song["n_en"] / per_song["n_tokens"]

display(per_song.describe().round(2))

print(f"Songs with no English tokens: {(per_song['n_en'] == 0).sum():,} ({(per_song['n_en'] == 0).mean():.2%})")
print(f"Songs with only English tokens (no Korean): {(per_song['n_ko'] == 0).sum():,}")

,n_tokens,n_ko,n_en,n_en_filler,en_share
count,7984.00,7984.00,7984.00,7984.00,7984.00
mean,209.36,159.51,49.29,3.71,0.17
std,97.33,67.71,81.59,11.59,0.24
min,10.00,0.00,0.00,0.00,0.00
25%,136.00,120.00,0.00,0.00,0.00
50%,183.00,149.00,9.00,0.00,0.05
75%,268.00,191.00,70.00,2.00,0.29
max,898.00,657.00,744.00,220.00,1.00


Songs with no English tokens: 3,322 (41.61%)
Songs with only English tokens (no Korean): 219


## 6. Export

### Output
- `data/interim/lyrics_clean.pkl`: one row per song with cleaned lyric lines and exclusion flags
- `data/interim/tokens.pkl`: one row per token (`song_key`, `line_no`, `token_no`, `token`, `lang`, `is_filler`)

In [10]:
lyric_columns = [
    "song_key", "lyrics_clean", "n_lines_raw", "n_lines_clean",
    "n_hangul", "n_latin", "n_kana", "n_han", "kana_ratio", "han_ratio",
    "flag_japanese", "flag_chinese", "flag_no_lyrics", "exclude_lyrics",
]
songs[lyric_columns].to_pickle(INTERIM_DIR / "lyrics_clean.pkl")
tokens.to_pickle(INTERIM_DIR / "tokens.pkl")

print(f"Saved {len(songs):,} songs to {INTERIM_DIR / 'lyrics_clean.pkl'}")
print(f"Saved {len(tokens):,} tokens to {INTERIM_DIR / 'tokens.pkl'}")

Saved 7,988 songs to ../data/interim/lyrics_clean.pkl
Saved 1,671,530 tokens to ../data/interim/tokens.pkl
